In [1]:
import os

GPU_NUM = 0
GPU_NUM = str(GPU_NUM)

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID" 
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_NUM 

# torch를 CUDA_VISIBLE_DEVICES 뒤에 import 해야 제대로 설정됨.
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f'CUDA Device Name: {torch.cuda.get_device_name(0)}, GPU NUM: {GPU_NUM}')
else: print(device)

torch.cuda.empty_cache() 

CUDA Device Name: NVIDIA RTX PRO 6000 Blackwell Server Edition, GPU NUM: 0


In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from utils.utils import *

allocate_dummy_gpu_memory(0.5)

Allocated dummy GPU memory: 512.00 MiB


In [6]:
def helper_out_activation(name: str | None):
    if name is None:
        return None

    target = name.lower()

    for attr_name in dir(nn):
        if attr_name.lower() == target:
            act_cls = getattr(nn, attr_name)
            break
    else:
        raise ValueError(f"Unsupported output activation: {name}")

    if not isinstance(act_cls, type) or not issubclass(act_cls, nn.Module):
        raise ValueError(f"Unsupported output activation: {name}")

    return act_cls()

In [7]:
import torch.nn as nn

t_act = helper_out_activation('leakyrelu')
t_act

LeakyReLU(negative_slope=0.01)

In [5]:
d = 4
chs = [64 * (2**i) for i in range(d)]
layers = []

layers.append((1, chs[0]))
for i in range(1, d):
    layers.append(
        (chs[i-1], chs[i])    
    )
layers.append((chs[-1], 1))

print(chs)
print(layers)

[64, 128, 256, 512]
[(1, 64), (64, 128), (128, 256), (256, 512), (512, 1)]


In [22]:
d = 3
chs = [64 * (2**i) for i in range(d)]
layers = []

layers.append((1, chs[0]))
for i in range(1, d):
    layers.append(
        (chs[i-1], chs[i])    
    )
layers.append((chs[-1], chs[-1]*2))

d_in_ch = chs[-1]*2
for i in reversed(range(d)):
    layers.append(
        (d_in_ch, chs[i])
    )
    d_in_ch = chs[i]
layers.append((chs[0], 1))

layers

[(1, 64),
 (64, 128),
 (128, 256),
 (256, 512),
 (512, 256),
 (256, 128),
 (128, 64),
 (64, 1)]